# Impact of Padding and Strides in Convolutional Layers

## Convolutional Neural Networks (CNNs)

Convolutional Neural Networks (CNNs) are deep learning models specifically designed to process data with a **grid-like structure**, such as images. Images can be viewed as 2D grids of pixel values, making CNNs highly effective for computer vision tasks.

CNNs automatically learn **spatial hierarchies of features**, starting from simple patterns (edges, textures) to complex structures (objects, shapes).

They form the backbone of many modern applications:

- Image classification  
- Object detection  
- Face recognition  
- Medical image analysis  
- Autonomous driving perception  

## Strides in Convolution

**Stride** defines how many pixels the filter moves at each step.

Example:

- **Stride = 1** → Filter moves one pixel at a time  
- **Stride = 2** → Filter jumps two pixels  

### Impact of Stride

Increasing stride:

* Reduces output size (downsampling effect)  
* Reduces computation  
* May lose fine details  

Decreasing stride:

* Preserves more spatial information  
* Increases computation  

---

## Padding in Convolution

**Padding** means adding extra pixels (usually zeros) around the image border.

Common types:

- **Valid Padding** → No padding  
- **Same Padding** → Pad so output size matches input size  

---

## Why Padding Matters

Without padding:

* Output shrinks after each convolution  
* Edge information is quickly lost  

With padding:

* Preserves spatial dimensions  
* Keeps border features  
* Allows deeper networks  

---


## Practical Interpretation

Modern CNNs carefully tune both to balance:

* Feature preservation  
* Computational efficiency  
* Network depth  

---

CNN performance and behavior are strongly influenced by stride and padding choices, making them critical design parameters.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

In [ ]:
# Load CIFAR-10 dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Normalize pixel values to be between 0 and 1
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Convert labels to categorical format (One-hot encoding)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

In [ ]:
import numpy as np


plt.figure(figsize=(10, 4))

indices = np.random.choice(len(X_train), 10)

for i, idx in enumerate(indices):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[idx])
    plt.axis("off")

plt.tight_layout()
plt.show()


### Default Stride ((1,1) for convolution and (2,2) for pooling) and Padding ('valid')

In [ ]:
# Define the CNN model
model = Sequential()

# First Convolutional Layer
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(MaxPooling2D((2, 2)))

# Second Convolutional Layer
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

# Flatten the output to feed into fully connected layers
model.add(Flatten())

# Fully Connected Layer
model.add(Dense(64, activation='relu'))

# Dropout Layer to avoid overfitting
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(10, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Summary of the model
model.summary()

In [ ]:
# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test), verbose=1)

In [ ]:
# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f'Test accuracy: {test_accuracy:.2f}')

In [ ]:
# Convert the history to a DataFrame for easy visualization
import pandas as pd
history_df = pd.DataFrame(history.history)

# Plot training and validation accuracy
plt.figure(figsize=(10, 6))
plt.plot(history_df['accuracy'], label='Training Accuracy')
plt.plot(history_df['val_accuracy'], linestyle='--', label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.show()

# Plot training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(history_df['loss'], label='Training Loss')
plt.plot(history_df['val_loss'], linestyle='--', label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

### Checking Different Stride Values

In [ ]:
experiments = [
    {"name": "Baseline (same padding, stride 1)",     "padding": "same",  "strides": (1,1)},
    {"name": "No padding (valid), stride 1",          "padding": "valid", "strides": (1,1)},
    {"name": "Same padding, stride 2",                "padding": "same",  "strides": (2,2)},
    {"name": "Valid padding, stride 2",               "padding": "valid", "strides": (2,2)},
    {"name": "Same padding, stride 3",                "padding": "same",  "strides": (3,3)},
]


In [ ]:
results = {}

# 3. Loop through experiments
for exp in experiments:
    print(f"\n=== Training: {exp['name']} ===")

    model = Sequential()

    # First Conv layer with experiment settings
    model.add(Conv2D(32, (3,3), activation='relu',
                     input_shape=(32,32,3),
                     padding=exp['padding'],
                     strides=exp['strides']))
    model.add(MaxPooling2D((2,2)))  # pooling remains the same for all

    # Second Conv layer
    model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
    model.add(MaxPooling2D((2,2)))

    # Flatten + Dense layers
    model.add(Flatten())
    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(10, activation='softmax'))

    # Compile
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


    history = model.fit(X_train, y_train, epochs=10, batch_size=64,
                        validation_split=0.1, verbose=2)

    # Evaluate on test set
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    results[exp['name']] = test_acc
    print(f"Test Accuracy: {test_acc:.4f}")

# 4. Show summary
print("\n=== Summary of Results ===")
for name, acc in results.items():
    print(f"{name}: {acc:.4f}")

## Insights from Stride & Padding Experiments

### Best Performing Configuration

**No padding (valid), stride 1 → Test Accuracy: 0.6705**

This configuration achieved the highest accuracy.

Possible reasons:

- **Valid padding** forces the network to use only real pixels (no artificial zeros)
- Encourages learning stronger, more localized features
- Slight reduction in spatial size may act as implicit regularization
- Less boundary noise compared to same padding

---

### Effect of Padding

**Same Padding (stride 1) → 0.6431**  
**Valid Padding (stride 1) → 0.6705**

Observation:

- Removing padding improved accuracy
- Indicates edge information was not critical for this dataset
- Zero-padding may have introduced unnecessary artifacts

Key takeaway:

Padding is not always beneficial — depends on dataset characteristics.

---

### Effect of Increasing Stride

**Stride 1 → Higher Accuracy**  
**Stride 2 → Moderate Drop**  
**Stride 3 → Significant Drop**

Accuracy trend:

- Stride 1 → Best feature preservation
- Stride 2 → Noticeable accuracy decrease
- Stride 3 → Strong performance degradation

Why this happens:

- Larger strides aggressively downsample feature maps
- Fine spatial details are lost early
- Important discriminative features may disappear

Key takeaway:

High strides trade accuracy for speed.

---

### Same vs Valid Padding at Higher Strides

Stride 2 Results:

- Same padding → 0.6205
- Valid padding → 0.6196

Observation:

- Minimal difference between padding types
- Stride impact dominates padding impact

Interpretation:

Once resolution drops, padding choice becomes less influential.

---

## Overall Conclusions

✔ **Stride has stronger impact than padding**  
✔ **Higher strides reduce accuracy due to information loss**  
✔ **Valid padding performed best at stride 1**  
✔ **Optimal configuration balances detail preservation & computation**

---

## Practical Implications for CNN Design

- Use **stride 1** when spatial precision is important
- Increase stride cautiously — consider pooling instead
- Padding strategy should be validated experimentally
- Model behavior is dataset-dependent

---
 **Most Important Finding:**  
For this task, preserving spatial detail (low stride) mattered more than padding strategy.